# M2→M3: From Gradient Descent to Multi-Layer Networks

**Act 1 (M2):** Where do gradients come from? We define a single neuron in sympy,
differentiate symbolically, lambdify into numpy, and run gradient descent by hand.

**Act 2 (M3):** We connect multiple neurons, trust backpropagation, and use the
pipeline's NumpyModel + CrossEntropyLoss to train a real MLP on Titanic.

## Act 1: A Single Neuron's Gradient

In [ ]:
import sympy as sp
import numpy as np

# Define symbols
w, b, x, t = sp.symbols('w b x t')
# Single neuron: y = sigmoid(w*x + b)
y = 1 / (1 + sp.exp(-(w*x + b)))
# MSE loss: L = (y - t)^2
L = (y - t)**2

# Symbolic gradients
dL_dw = sp.diff(L, w)
dL_db = sp.diff(L, b)

print('dL/dw =')
sp.simplify(dL_dw)

In [ ]:
print('dL/db =')
sp.simplify(dL_db)

In [ ]:
# Lambdify into numpy functions
grad_w_fn = sp.lambdify([w, b, x, t], dL_dw, 'numpy')
grad_b_fn = sp.lambdify([w, b, x, t], dL_db, 'numpy')

# Test on one data point
w_val, b_val = 0.5, 0.0
x_val, t_val = 1.0, 0.0

print(f'grad_w = {grad_w_fn(w_val, b_val, x_val, t_val):.6f}')
print(f'grad_b = {grad_b_fn(w_val, b_val, x_val, t_val):.6f}')

In [ ]:
# Manual gradient descent — single neuron learning y = 2x
from pipeline.adapters.numpy_adapter import NumpyModel, NumpyOptimizer
from pipeline.training.optimizers import SGD
from pipeline.protocols import Parameter

# Initialize parameters
W_data = np.array([[0.5]])
b_data = np.array([0.0])
model = NumpyModel([(W_data, b_data)], activation='relu')
opt = NumpyOptimizer(model.parameters(), SGD(lr=0.1))

# Training data: y = 2x
X = np.array([[1.0], [2.0], [3.0], [4.0]])
y = np.array([[2.0], [4.0], [6.0], [8.0]])

losses = []
for epoch in range(50):
    epoch_loss = 0
    for i in range(len(X)):
        x_i = X[i:i+1]
        y_i = y[i:i+1]
        pred = model.forward(x_i)
        loss_val = float(np.mean((pred - y_i)**2))
        epoch_loss += loss_val

        # Manual gradient: dL/dpred = 2*(pred - y_i)
        dL_dpred = 2 * (pred - y_i)
        model.backward(dL_dpred)

        opt.step()
        opt.zero_grad()

    losses.append(epoch_loss / len(X))

print(f'Final loss: {losses[-1]:.6f}')
W_learned = list(model.parameters())[0].data[0, 0]
b_learned = list(model.parameters())[1].data[0]
print(f'Learned W: {W_learned:.4f} (target: 2.0)')
print(f'Learned b: {b_learned:.4f} (target: 0.0)')

In [ ]:
# Plot the loss curve
import matplotlib.pyplot as plt
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Single Neuron: Gradient Descent from First Principles')
plt.show()

**What we just did:** We derived the gradient formulas by hand (sympy),
converted them to numpy (lambdify), ran gradient descent manually,
and verified the neuron learned y = 2x.

**Now:** Let's do the same thing for a multi-layer network on Titanic.
But this time, we let backpropagation handle the gradients.

## Act 2: Multi-Layer Network on Titanic

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, LabelEncoder

from pipeline.config import Config
from pipeline.data.csv_source import CsvDataSource
from pipeline.data.split import train_test_split
from pipeline.evaluation.metrics import Metrics, accuracy
from pipeline.hooks.progress import ProgressHook
from pipeline.pipeline import BasePipeline, PipelineState
from pipeline.adapters.numpy_adapter import NumpyModel, NumpyOptimizer
from pipeline.training.losses import CrossEntropyLoss
from pipeline.training.optimizers import Adam

# Fetch and preprocess Titanic
titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
df = titanic.data.copy()
df['survived'] = titanic.target.astype(int)
cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df = df[cols].copy()
df['sex'] = LabelEncoder().fit_transform(df['sex'])
df['embarked'] = LabelEncoder().fit_transform(df['embarked'].astype(str))
for c in ['age', 'fare']:
    df[c] = df[c].fillna(df[c].median())
df['embarked'] = df['embarked'].fillna(0)

# Scale features
scaler = StandardScaler()
feature_cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X = scaler.fit_transform(df[feature_cols].astype(float))
scaled_df = pd.DataFrame(X, columns=feature_cols)
scaled_df['survived'] = df['survived'].astype(int)

import tempfile, os
_tmpdir = tempfile.mkdtemp()
csv_path = os.path.join(_tmpdir, 'titanic.csv')
scaled_df.to_csv(csv_path, index=False)
print(f'Data: {scaled_df.shape[0]} rows, {len(feature_cols)} features')

In [ ]:
config = Config(
    batch_size=32, num_epochs=100, train_ratio=0.8, learning_rate=0.01,
    seed=42, output_dir=f'{_tmpdir}/output', task_name='titanic-numpy-mlp',
)


class TitanicNumpyMLP(BasePipeline):
    def load_data(self, state: PipelineState) -> None:
        source = CsvDataSource(
            csv_path, batch_size=config.batch_size,
            target_column='survived', shuffle=True, seed=config.seed,
        )
        train, val = train_test_split(source, train_ratio=config.train_ratio)
        state.data_stream = train
        state.val_data_stream = val

    def build_model(self, state: PipelineState) -> None:
        rng = np.random.default_rng(config.seed)
        # 7 input features -> 16 hidden -> 2 output classes
        W1 = rng.standard_normal((16, 7)) * np.sqrt(2.0 / 7)
        b1 = np.zeros(16)
        W2 = rng.standard_normal((2, 16)) * np.sqrt(2.0 / 16)
        b2 = np.zeros(2)

        state.model = NumpyModel([(W1, b1), (W2, b2)], activation='relu')
        state.loss_fn = CrossEntropyLoss(model=state.model)
        state.optimizer = NumpyOptimizer(
            state.model.parameters(), Adam(lr=config.learning_rate),
        )
        state.metrics = Metrics(accuracy=accuracy)

    def evaluate(self, state: PipelineState) -> None:
        state.model.eval_mode()
        preds, trues = [], []
        for batch in state.val_data_stream:
            logits = np.asarray(state.model.forward(batch.inputs))
            preds.append(np.argmax(logits, axis=1))
            trues.append(np.asarray(batch.targets))
        y_pred = np.concatenate(preds)
        y_true = np.concatenate(trues).astype(np.int64)
        state.metrics.compute(y_true, y_pred)

    def export(self, state: PipelineState) -> None:
        state.predictions = np.array([0])

In [ ]:
pipeline = TitanicNumpyMLP(config)
pipeline.add_hook(ProgressHook())
state = pipeline.run('train')

print(f'Accuracy:    {state.metrics["accuracy"]:.4f}')
print(f'Final loss:  {state.history["loss"][-1]:.4f}')
print(f'Epochs:      {len(state.history["loss"])}')

## Compare: M1 (sklearn) vs M3 (numpy MLP)

| | M1 sklearn | M3 numpy MLP |
|---|---|---|
| Model | LogisticRegression | 2-layer MLP (7→16→2) |
| Training | `.fit()` one call | 100 epochs per-batch GD |
| Optimizer | LBFGS (internal) | Adam (lr=0.01) |
| Loss | log-loss (internal) | CrossEntropyLoss |
| Parameters | None exposed | 4 Parameters (W1,b1,W2,b2) |
| Gradient | sklearn internal | manual backprop in `model.backward()` |

**Key insight:** The same pipeline structure ran both. The sklearn path overrode
`train()`; the numpy path used the default `TrainLoop`. The protocol design works.